> **ARCHIVE — superseded.** This notebook is kept as a record of the research path, not as working code. Credentials were removed and outputs cleared. Results in it may rest on defects documented in `docs/RESEARCH_LOG.md` (stage: v1, LLM-named 82-name universe). Comments are partly in Vietnamese. The maintained implementation is `src/memescan/`.

# Meme Scanner v1 — LEAN (no LLM in the decision path)

Strip-down cua pipeline cu. Nguyen tac:
- **2 layers**: Layer-1 SCAN (1x/ngay, big universe, re tien) -> watchlist nho.
  Layer-2 CONFIRM (moi 15 phut, CHI tren watchlist) -> entry.
- **Cascade re-truoc**: filter nao loai duoc nhieu + re thi chay truoc; LLM di CUOI cung (tiet kiem token).
- **LLM KHONG nam trong decision path** (CFG["LLM"]["enabled"]=False). De backtest, de finetune.
- **It dependency**: hard dep = Alpaca. Moi thu khac optional + degrade, khong crash.
- Fundamentals gate = **strong OR recovering** (khong phai chi "yeu").
- Exit co **no-meme 3 ngay** + hard stop + ATR trail + thesis monitor.

Reuse cache cu: `meme_cache/bt_daily.parquet`, `short_interest.parquet`, `fundamentals_*.parquet`.

In [ ]:
# Cell 1 - CONFIG v1 (tat ca tham so o MOT cho)
import os, time, json, logging
from pathlib import Path
from datetime import datetime, timezone, timedelta
import numpy as np, pandas as pd, requests

logging.basicConfig(level=logging.INFO, format="%(asctime)s | %(levelname)s | %(message)s")
log = logging.getLogger("v1")
CACHE = Path("./meme_cache"); CACHE.mkdir(exist_ok=True)

CFG = {
    "alpaca": {
        "key":    os.getenv("ALPACA_API_KEY", ""),
        "secret": os.getenv("ALPACA_SECRET_KEY", ""),
        "paper":  "https://paper-api.alpaca.markets/v2",
        "data":   "https://data.alpaca.markets/v2",
    },
    # ---------- LAYER 1: SCAN (1x/ngay, big universe) ----------
    "L1": {
        "min_price":        1.0,
        "min_dollar_vol":   3e6,     # thanh khoan: $vol TB20 >= 3M (vao/ra duoc)
        "mcap_floor":       1e8,
        "mcap_ceiling":     3e10,
        # -- dinh nghia TRENDING (comment 4) --
        "min_mentions":     10,      # san tuyet doi, tranh nhieu tick nho
        "trend_growth":     1.5,     # mentions / mentions_24h_ago >= 1.5
        "trend_rank_jump":  20,      # HOAC nhay >= 20 bac tren bang ApeWisdom
        # -- short fuel: ADAPTIVE percentile, khong fix cung --
        "short_pct_q":      0.60,
        "dtc_q":            0.60,
        # -- fundamentals: strong OR recovering (comment 1) --
        "fund_mode":        "strong_or_recovering",
        "watchlist_max":    40,      # tran watchlist -> gioi han query cua Layer 2
        # -- BUG FIX: khong co SI tai ngay do thi lam gi? --
        # "reject" = coi nhu fail  -> AM THAM xoa sach giai doan truoc khi co SI history (2021-2024!)
        # "allow"  = khong chan    -> gate chi loc khi THUC SU co du lieu   <-- DUNG
        "si_missing":       "allow",
    },
    # ---------- LAYER 2: CONFIRM (intraday, chi tren watchlist) ----------
    "L2": {
        "interval_min":         15,
        "rvol_min":             3.0,   # RVOL time-of-day adjusted
        "require_above_vwap":   True,
        "require_follow_through": True,
        "max_ext_ma20":         0.50,  # exhaustion cap (data: .50 la diem ngot)
        # -- BUG FIX: v1 lo BO 2 filter xu huong cua prototype -> mua ca ten DUOI MA20 --
        "require_above_ma20":   True,  # phai TREN MA20 (ext > 0)
        "mom5_min":             0.05,  # momentum 5 phien > 5%
    },
    # ---------- ENTRY / EXIT ----------
    "EXIT": {
        "hard_stop":    0.08,   # -8% tu entry
        # comment 9. ABLATION (Cell 5b) cho thay quy tac nay TON tien tren mau day:
        #   TAT = +351k | 3 ngay = +311k | 5 ngay = +324k | 7 ngay = +341k
        # -> mac dinh TAT; bat len de tu kiem chung.
        "use_no_meme":  False,
        "no_meme_days": 3,
        "no_meme_mfe":  0.10,
        "atr_win":      14,
        "atr_mult":     3.0,    # trailing = peak - 3*ATR
        "ma_exit":      10,     # gay MA10 sau khi da loi -> thoat
        "max_hold":     60,
        "thesis_monitor": True, # short-fuel phai CON dung moi ngay hold
    },
    "SIZE": {"alloc": 10000.0, "max_positions": 10, "start_capital": 100000.0},
    # ---------- LLM: TAT trong v1 (comment 8) ----------
    "LLM": {
        "enabled":  False,          # v1 KHONG dung LLM de quyet dinh
        "when":     "once_per_day", # neu bat: 1 lan/ngay tren top_n, cache theo (ticker, ngay)
        "top_n":    5,
    },
    "http_headers": {
        "User-Agent": "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 "
                      "(KHTML, like Gecko) Chrome/124.0.0.0 Safari/537.36",
        "Accept": "application/json, text/plain, */*",
    },
}
ALP_H = {"APCA-API-KEY-ID": CFG["alpaca"]["key"], "APCA-API-SECRET-KEY": CFG["alpaca"]["secret"]}

def load_df(p):
    p = Path(p)
    if p.exists():
        try: return pd.read_parquet(p)
        except Exception: pass
    pk = Path(str(p) + ".pkl")
    return pd.read_pickle(pk) if pk.exists() else None

def save_df(df, p):
    try: df.to_parquet(p, index=False)
    except Exception: df.to_pickle(Path(str(p) + ".pkl"))

print("v1 config OK | LLM in decision path:", CFG["LLM"]["enabled"])

## Layer 1 - SCAN (1x/ngay)

**Cascade re-truoc, dat-sau** (comment 6). Moi stage log lai: con bao nhieu ten + ton bao nhieu API call.
Muc tieu: cat that nhieu ten bang stage RE, de stage DAT chi con chay tren so it.

In [ ]:
# Cell 2 - Layer 1: cascade + funnel accounting
FUNNEL = []

def _log_stage(name, n_in, n_out, api_calls, cost_note):
    FUNNEL.append({"stage": name, "in": n_in, "out": n_out,
                   "removed": n_in - n_out, "api_calls": api_calls, "cost": cost_note})
    log.info("L1 %-22s %5d -> %5d  (-%d)  api=%s", name, n_in, n_out, n_in - n_out, api_calls)

# --- Stage 1: tradable universe (1 API call cho CA THI TRUONG) ---
def stage_tradable():
    try:
        r = requests.get(CFG["alpaca"]["paper"] + "/assets",
                         params={"status": "active", "asset_class": "us_equity"},
                         headers=ALP_H, timeout=30)
        r.raise_for_status()
        syms = [a["symbol"] for a in r.json()
                if a.get("tradable") and a.get("exchange") in ("NYSE", "NASDAQ", "AMEX", "ARCA", "BATS")]
        _log_stage("1 tradable (Alpaca)", 0, len(syms), 1, "1 call / toan bo market")
        return syms
    except Exception as e:
        log.warning("Alpaca /assets loi: %s", e); return []

# --- Stage 2: TRENDING tu ApeWisdom (1-3 call cho CA BANG) ---
def stage_trending(tradable_set):
    """Dinh nghia TRENDING (comment 4): can san mentions tuyet doi, VA
    (tang toc >= trend_growth  HOAC  nhay >= trend_rank_jump bac)."""
    L1 = CFG["L1"]; rows = []; calls = 0
    for flt in ("all-stocks", "wallstreetbets"):
        for page in (1, 2):
            try:
                r = requests.get(f"https://apewisdom.io/api/v1.0/filter/{flt}/page/{page}",
                                 headers=CFG["http_headers"], timeout=10); calls += 1
                if r.status_code != 200: break
                for it in r.json().get("results", []):
                    rows.append({
                        "ticker": (it.get("ticker") or "").upper(),
                        "mentions": float(it.get("mentions") or 0),
                        "mentions_24h": float(it.get("mentions_24h_ago") or 0),
                        "rank": float(it.get("rank") or 9999),
                        "rank_24h": float(it.get("rank_24h_ago") or 9999),
                        "upvotes": float(it.get("upvotes") or 0),
                    })
                time.sleep(0.3)
            except Exception as e:
                log.debug("apewisdom %s p%d: %s", flt, page, e)
    if not rows:
        _log_stage("2 trending (ApeWisdom)", 0, 0, calls, "0 - API trong")
        return pd.DataFrame(columns=["ticker", "mentions", "growth", "rank_jump"])
    df = pd.DataFrame(rows).drop_duplicates("ticker")
    n_in = len(df)
    df["growth"] = (df["mentions"] + 1) / (df["mentions_24h"] + 1)
    df["rank_jump"] = df["rank_24h"] - df["rank"]           # duong = leo bang
    trend = df[(df["mentions"] >= L1["min_mentions"]) &
               ((df["growth"] >= L1["trend_growth"]) | (df["rank_jump"] >= L1["trend_rank_jump"]))]
    if tradable_set:
        trend = trend[trend["ticker"].isin(tradable_set)]
    _log_stage("2 trending (ApeWisdom)", n_in, len(trend), calls, "~4 call / ca bang")
    return trend.reset_index(drop=True)

# --- Stage 3: thanh khoan + gia (1 call/ten, chi tren ten da trending) ---
def stage_liquidity(tickers, lookback=20):
    L1 = CFG["L1"]; keep = []; calls = 0
    start = (datetime.now(timezone.utc) - timedelta(days=lookback * 3)).strftime("%Y-%m-%d")
    for sym in tickers:
        try:
            r = requests.get(f"{CFG['alpaca']['data']}/stocks/{sym}/bars",
                             params={"timeframe": "1Day", "start": start, "limit": lookback + 5,
                                     "feed": "iex", "adjustment": "split"},
                             headers=ALP_H, timeout=10); calls += 1
            bars = r.json().get("bars") or [] if r.status_code == 200 else []
            if len(bars) < lookback: continue
            c = np.array([b["c"] for b in bars], float); v = np.array([b["v"] for b in bars], float)
            dvol = float(np.mean(c[-lookback:] * v[-lookback:]))
            if c[-1] >= L1["min_price"] and dvol >= L1["min_dollar_vol"]:
                keep.append({"ticker": sym, "price": c[-1], "dollar_vol": dvol})
            time.sleep(0.08)
        except Exception as e:
            log.debug("liq %s: %s", sym, e)
    _log_stage("3 liquidity (Alpaca)", len(tickers), len(keep), calls, "1 call/ten")
    return pd.DataFrame(keep)

print("Layer-1 stages 1-3 ready")

In [ ]:
# Cell 3 - Layer 1 stage 4-5: short fuel (adaptive) + fundamentals (strong OR recovering)
SI = load_df(CACHE / "short_interest.parquet")
si_map = {t: g.sort_values("date") for t, g in SI.groupby("ticker")} if SI is not None and len(SI) else {}
SI_PUB_LAG_D = 12   # FINRA cong bo tre ~12 ngay -> point-in-time, tranh look-ahead

def si_asof(sym, d):
    g = si_map.get(sym)
    if g is None: return None
    dd = pd.to_datetime(d)
    if getattr(dd, "tzinfo", None) is not None: dd = dd.tz_localize(None)
    av = g[g["date"] + pd.Timedelta(days=SI_PUB_LAG_D) <= dd]
    if len(av) == 0: return None
    last = av.iloc[-1]
    rising = bool(len(av) >= 2 and last["interest"] > av.iloc[-2]["interest"])
    return (last.get("dtc", np.nan), rising)

def stage_short_fuel(df, asof=None):
    """ADAPTIVE: nguong = percentile cua chinh cross-section hom nay (khong fix cung)."""
    if df is None or len(df) == 0 or not si_map:
        _log_stage("4 short fuel", len(df) if df is not None else 0,
                   len(df) if df is not None else 0, 0, "skip - khong co SI")
        return df
    asof = asof or datetime.now()
    rows = []
    for t in df["ticker"]:
        si = si_asof(t, asof)
        rows.append({"ticker": t, "dtc": (si[0] if si else np.nan),
                     "si_rising": (si[1] if si else False)})
    s = pd.DataFrame(rows)
    d = df.merge(s, on="ticker", how="left")
    thr = np.nanpercentile(d["dtc"].dropna(), CFG["L1"]["dtc_q"] * 100) if d["dtc"].notna().any() else np.nan
    keep = d[(d["dtc"] >= thr) | (d["si_rising"])] if not np.isnan(thr) else d
    _log_stage("4 short fuel (adaptive)", len(df), len(keep), 0,
               f"0 call (cache) | dtc_thr={thr:.2f}" if not np.isnan(thr) else "0 call")
    return keep.reset_index(drop=True)

def classify_fundamentals(pm, rg, eg):
    """comment 1: chap nhan STRONG hoac RECOVERING; loai ten dang xau di.
    pm=profit margin, rg=revenue growth, eg=earnings growth."""
    pm = np.nan if pm is None else pm; rg = np.nan if rg is None else rg; eg = np.nan if eg is None else eg
    if (not np.isnan(pm) and pm > 0.05) and (not np.isnan(rg) and rg > 0):
        return "strong"
    if (not np.isnan(rg) and rg > 0.10) or (not np.isnan(eg) and eg > 0):
        return "recovering"
    if (not np.isnan(rg) and rg < 0) and (not np.isnan(pm) and pm < 0):
        return "deteriorating"
    return "unknown"

def stage_fundamentals(df, fund_df=None):
    if df is None or len(df) == 0: return df
    if fund_df is None:
        cands = sorted(CACHE.glob("fundamentals_*.parquet"))
        fund_df = load_df(cands[-1]) if cands else None
    if fund_df is None or "ticker" not in getattr(fund_df, "columns", []):
        _log_stage("5 fundamentals", len(df), len(df), 0, "skip - khong co cache")
        return df
    f = fund_df.copy()
    for c in ("profit_margin", "revenue_growth", "earnings_growth"):
        if c not in f: f[c] = np.nan
    f["fund_state"] = [classify_fundamentals(a, b, c) for a, b, c in
                       zip(f["profit_margin"], f["revenue_growth"], f["earnings_growth"])]
    d = df.merge(f[["ticker", "fund_state", "market_cap", "float_shares", "short_pct_float"]],
                 on="ticker", how="left")
    ok = d[d["fund_state"].isin(["strong", "recovering", "unknown"])]      # loai 'deteriorating'
    if "market_cap" in ok:
        ok = ok[(ok["market_cap"].isna()) |
                ((ok["market_cap"] > CFG["L1"]["mcap_floor"]) & (ok["market_cap"] < CFG["L1"]["mcap_ceiling"]))]
    _log_stage("5 fundamentals (cache)", len(df), len(ok), 0, "0 call (cache 1x/ngay)")
    return ok.reset_index(drop=True)

def build_watchlist():
    FUNNEL.clear()
    tradable = set(stage_tradable())
    trend = stage_trending(tradable)
    if len(trend) == 0: return pd.DataFrame()
    liq = stage_liquidity(trend["ticker"].tolist())
    if len(liq) == 0: return pd.DataFrame()
    d = liq.merge(trend, on="ticker", how="left")
    d = stage_short_fuel(d)
    d = stage_fundamentals(d)
    d = d.sort_values("growth", ascending=False).head(CFG["L1"]["watchlist_max"])
    _log_stage("6 watchlist cap", len(d), len(d), 0, f"tran {CFG['L1']['watchlist_max']}")
    return d.reset_index(drop=True)

print("Layer-1 ready. Chay: wl = build_watchlist(); pd.DataFrame(FUNNEL)")

## Layer 2 - CONFIRM (intraday, moi 15 phut, CHI tren watchlist)

Layer 1 da cat universe xuong <=40 ten -> Layer 2 chi ton **40 call / 15 phut** (~26 lan/phien = ~1000 call/ngay),
thay vi quet ca thi truong moi 15 phut (comment 4: "scanner giup giam data query cua confirmation").

**RVOL intraday = volume luy ke hom nay / TB volume luy ke CUNG THOI DIEM trong ngay** (time-of-day adjusted),
giong cac intraday strategy hien co - khong so volume 10:00 voi ca ngay.

In [ ]:
# Cell 4 - Layer 2: intraday confirmation + ENTRY signal
def intraday_snapshot(sym, lookback_days=20):
    """Tra ve dict: rvol (time-of-day adjusted), price, vwap, prev_close, ext_ma20."""
    try:
        start = (datetime.now(timezone.utc) - timedelta(days=lookback_days * 2)).strftime("%Y-%m-%d")
        r = requests.get(f"{CFG['alpaca']['data']}/stocks/{sym}/bars",
                         params={"timeframe": "15Min", "start": start, "limit": 10000, "feed": "iex"},
                         headers=ALP_H, timeout=15)
        if r.status_code != 200: return None
        b = r.json().get("bars") or []
        if len(b) < 30: return None
        m = pd.DataFrame(b).rename(columns={"t": "ts", "o": "o", "h": "h", "l": "l", "c": "c", "v": "v"})
        m["ts"] = pd.to_datetime(m["ts"]); m["day"] = m["ts"].dt.date
        m["slot"] = m.groupby("day").cumcount()                      # thu tu bar trong ngay
        m["cum_v"] = m.groupby("day")["v"].cumsum()
        today = m["day"].iloc[-1]; slot_now = int(m["slot"].iloc[-1])
        hist = m[(m["day"] < today) & (m["slot"] == slot_now)]       # CUNG slot cac ngay truoc
        base = hist["cum_v"].tail(lookback_days).mean()
        cur = m[m["day"] == today]
        rvol = float(cur["cum_v"].iloc[-1] / base) if base and base > 0 else np.nan
        vwap = float((cur["c"] * cur["v"]).cumsum().iloc[-1] / max(cur["v"].cumsum().iloc[-1], 1))
        dayc = m.groupby("day")["c"].last()
        prev_close = float(dayc.iloc[-2]) if len(dayc) >= 2 else np.nan
        ma20 = float(dayc.tail(21).iloc[:-1].mean()) if len(dayc) >= 21 else np.nan
        px = float(cur["c"].iloc[-1])
        return {"ticker": sym, "price": px, "vwap": vwap, "rvol": rvol,
                "prev_close": prev_close,
                "ext_ma20": (px / ma20 - 1) if ma20 and not np.isnan(ma20) else np.nan}
    except Exception as e:
        log.debug("intraday %s: %s", sym, e); return None

def confirm_entry(snap):
    """ENTRY = tat ca dieu kien Layer-2 phai dung CUNG LUC."""
    if snap is None: return False, "no-data"
    L2 = CFG["L2"]; why = []
    if not (snap["rvol"] and snap["rvol"] >= L2["rvol_min"]):        return False, "rvol thap"
    why.append(f"RVOL={snap['rvol']:.1f}")
    if L2["require_above_vwap"] and not (snap["price"] > snap["vwap"]):   return False, "duoi VWAP"
    why.append("tren VWAP")
    if L2["require_follow_through"] and not (snap["price"] > snap["prev_close"]): return False, "khong follow-through"
    why.append("follow-through")
    if not np.isnan(snap["ext_ma20"]) and snap["ext_ma20"] > L2["max_ext_ma20"]:  return False, "da du dinh"
    why.append(f"ext={snap['ext_ma20']:+.0%}")
    return True, " | ".join(why)

def layer2_cycle(watchlist):
    """1 vong 15 phut: chi query cac ten trong watchlist."""
    out = []
    for sym in watchlist:
        s = intraday_snapshot(sym)
        ok, why = confirm_entry(s)
        if s: out.append({**s, "ENTRY": ok, "why": why})
        time.sleep(0.1)
    df = pd.DataFrame(out)
    log.info("L2 cycle: %d ten query | %d ENTRY", len(watchlist), int(df["ENTRY"].sum()) if len(df) else 0)
    return df

print(f"Layer-2 ready | cost = {CFG['L1']['watchlist_max']} call / {CFG['L2']['interval_min']} phut")

## Entry / Exit dac ta (comment 9, 11)

**ENTRY** (Layer 2, danh gia moi 15 phut tren watchlist):
`RVOL_tod >= 3` AND `price > VWAP phien` AND `price > prev_close` AND `ext_MA20 <= +50%`
-> vao bang MARKET o bar 15 phut ke tiep. Size = $10k co dinh, toi da 10 vi the.

**EXIT** (uu tien tu tren xuong, check moi bar):
1. **Hard stop** `-8%` tu gia vao.
2. **No-meme 3 ngay**: sau 3 phien, neu chua tung chay `>= +10%` VA dang <= gia vao -> thoat.
3. **Thesis monitor**: short-fuel mat -> thoat (khong doi gia).
4. **ATR trailing**: `peak - 3*ATR(14)`.
5. **Gay MA10** sau khi da co lai.
6. **Max hold** 60 phien.

In [ ]:
# Cell 5 - BACKTEST v1 tren daily cache (proxy cua rule intraday)
bt = load_df(CACHE / "bt_daily.parquet")
assert bt is not None and len(bt), "Chua co bt_daily.parquet - chay Cell 8.1 cua notebook chinh."
E = CFG["EXIT"]; S = CFG["SIZE"]

def _atr(h, l, c, w):
    n = len(c); tr = np.zeros(n)
    for i in range(1, n):
        tr[i] = max(h[i]-l[i], abs(h[i]-c[i-1]), abs(l[i]-c[i-1]))
    a = np.full(n, np.nan)
    for i in range(w, n): a[i] = tr[i-w+1:i+1].mean()
    return a

def _si_ok(sym, d):
    if not si_map: return True
    si = si_asof(sym, d)
    if not si:                       # KHONG co du lieu tai ngay do
        return CFG["L1"]["si_missing"] == "allow"
    dtc, rising = si
    return bool((dtc is not None and not (isinstance(dtc, float) and np.isnan(dtc)) and dtc >= 2.0) or rising)

def backtest_v1(bt_df):
    """Entry/exit v1. NO LOOK-AHEAD: quyet dinh vao chi dung bar <= i."""
    trades = []
    for sym, g in bt_df.groupby("ticker"):
        g = g.sort_values("date").reset_index(drop=True)
        if len(g) < 40: continue
        c = g["close"].values.astype(float); v = g["vol"].values.astype(float)
        h = g["high"].values.astype(float); l = g["low"].values.astype(float)
        n = len(g); atr = _atr(h, l, c, E["atr_win"])
        i = 21
        while i < n - 1:
            rvol = v[i] / v[i-20:i].mean() if v[i-20:i].mean() > 0 else 0
            ma20 = c[i-20:i].mean(); ext = c[i]/ma20 - 1 if ma20 > 0 else 0
            mom5 = c[i]/c[i-5] - 1
            vwap_px = (h[i] + l[i] + c[i]) / 3        # proxy VWAP tren daily bar
            ok = (rvol >= CFG["L2"]["rvol_min"] and c[i] > vwap_px and c[i] > c[i-1]
                  and ext <= CFG["L2"]["max_ext_ma20"])
            if CFG["L2"]["require_above_ma20"]: ok = ok and (c[i] > ma20)   # tren MA20
            if CFG["L2"]["mom5_min"] is not None: ok = ok and (mom5 > CFG["L2"]["mom5_min"])
            ok = ok and _si_ok(sym, g["date"].iloc[i])
            if ok:
                entry = c[i]; peak = entry; xi = min(i + E["max_hold"], n - 1); reason = "max_hold"
                for j in range(i + 1, xi + 1):
                    mfe = c[i+1:j+1].max() / entry - 1 if j > i else 0
                    if c[j] <= entry * (1 - E["hard_stop"]):
                        xi, reason = j, "hard_stop"; break
                    if E["use_no_meme"] and (j - i) >= E["no_meme_days"] and mfe < E["no_meme_mfe"] and c[j] <= entry:
                        xi, reason = j, "no_meme_3d"; break
                    if E["thesis_monitor"] and not _si_ok(sym, g["date"].iloc[j]):
                        xi, reason = j, "thesis_lost"; break
                    peak = max(peak, c[j])
                    if c[j] <= peak - E["atr_mult"] * (atr[j] if not np.isnan(atr[j]) else 0):
                        xi, reason = j, "atr_trail"; break
                    if c[j] > entry and c[j] < c[max(0, j-E["ma_exit"]+1):j+1].mean():
                        xi, reason = j, "ma_break"; break
                trades.append({"ticker": sym, "entry_date": g["date"].iloc[i], "exit_date": g["date"].iloc[xi],
                               "entry_px": entry, "exit_px": c[xi], "ret": c[xi]/entry - 1,
                               "days": xi - i, "rvol_in": rvol, "ext_in": ext,
                               "mom5_in": mom5, "exit_reason": reason})
                i = xi + 1; continue
            i += 1
    return pd.DataFrame(trades).sort_values("entry_date").reset_index(drop=True) if trades else pd.DataFrame()

def portfolio(td):
    rows, opens = [], []
    for _, t in td.iterrows():
        opens = [d for d in opens if d > t["entry_date"]]
        if len(opens) >= S["max_positions"]: continue
        opens.append(t["exit_date"]); rows.append({**t.to_dict(), "pnl": S["alloc"] * t["ret"]})
    tk = pd.DataFrame(rows)
    if len(tk):
        tk = tk.sort_values("exit_date").reset_index(drop=True)
        tk["equity"] = S["start_capital"] + tk["pnl"].cumsum()
    return tk

v1_trades = backtest_v1(bt)
v1_taken = portfolio(v1_trades) if len(v1_trades) else pd.DataFrame()
print("v1 signals:", len(v1_trades), "| taken:", len(v1_taken))

In [ ]:
# Cell 5b - COVERAGE GUARD + ABLATION (tai sao ket qua doi khi doi 1 tham so)
# 1) COVERAGE: neu SI history ngan hon bt -> gate "reject" se AM THAM xoa ca giai doan.
bt_lo, bt_hi = pd.to_datetime(bt["date"]).min(), pd.to_datetime(bt["date"]).max()
print(f"bt_daily   : {bt_lo:%Y-%m-%d} -> {bt_hi:%Y-%m-%d} | {bt['ticker'].nunique()} ticker")
if si_map:
    si_lo = min(g["date"].min() for g in si_map.values())
    si_hi = max(g["date"].max() for g in si_map.values())
    print(f"short_int  : {si_lo:%Y-%m-%d} -> {si_hi:%Y-%m-%d} | {len(si_map)} ticker")
    lost = (pd.to_datetime(si_lo).tz_localize('UTC') - bt_lo).days
    if lost > 30:
        print(f"  !! SI thieu {lost} ngay dau. si_missing='{CFG['L1']['si_missing']}'"
              f" -> {'OK (khong chan)' if CFG['L1']['si_missing']=='allow' else 'DANG XOA ca giai doan nay!'}")
else:
    print("short_int  : TRONG -> gate tu tat")

# 2) ABLATION: bat/tat tung thanh phan de thay cai nao gay ra chenh lech
def ablate(**over):
    L2_bak = dict(CFG["L2"]); E_bak = dict(CFG["EXIT"]); L1_bak = dict(CFG["L1"])
    CFG["L2"].update({k: v for k, v in over.items() if k in CFG["L2"]})
    CFG["EXIT"].update({k: v for k, v in over.items() if k in CFG["EXIT"]})
    CFG["L1"].update({k: v for k, v in over.items() if k in CFG["L1"]})
    td = backtest_v1(bt); tk = portfolio(td) if len(td) else pd.DataFrame()
    CFG["L2"].clear(); CFG["L2"].update(L2_bak)
    CFG["EXIT"].clear(); CFG["EXIT"].update(E_bak)
    CFG["L1"].clear(); CFG["L1"].update(L1_bak)
    if len(tk) == 0: return {"n": 0, "win": np.nan, "exp": np.nan, "pf": np.nan, "pnl": 0, "years": ""}
    r = tk["ret"]; w = tk[r > 0]; l = tk[r <= 0]
    pf = w["pnl"].sum()/abs(l["pnl"].sum()) if len(l) and l["pnl"].sum() != 0 else np.inf
    yr = sorted(pd.to_datetime(tk["entry_date"]).dt.year.unique())
    return {"n": len(tk), "win": (r > 0).mean(), "exp": r.mean(), "pf": pf,
            "pnl": tk["pnl"].sum(), "years": f"{yr[0]}-{yr[-1]}"}

rows = []
for lbl, over in [
    ("v1 cu (si=reject, ko trend)", dict(si_missing="reject", require_above_ma20=False, mom5_min=None, use_no_meme=True)),
    ("+ si_missing=allow",          dict(si_missing="allow",  require_above_ma20=False, mom5_min=None, use_no_meme=True)),
    ("+ tra lai trend filter",      dict(si_missing="allow",  require_above_ma20=True,  mom5_min=0.05, use_no_meme=True)),
    ("+ tat no_meme  (= v1 moi)",   dict(si_missing="allow",  require_above_ma20=True,  mom5_min=0.05, use_no_meme=False)),
]:
    rows.append({"config": lbl, **ablate(**over)})
ab = pd.DataFrame(rows)
ab["win"] = ab["win"].map(lambda x: f"{x:.1%}" if pd.notna(x) else "-")
ab["exp"] = ab["exp"].map(lambda x: f"{x:+.2%}" if pd.notna(x) else "-")
ab["pnl"] = ab["pnl"].map(lambda x: f"{x:,.0f}")
print("\n" + ab.to_string(index=False))
print("\n-> Chenh lech LON nhat den tu si_missing: 'reject' xoa het 2021-2024 vi SI chi co ~12 thang.")


In [ ]:
# Cell 6 - REPORT RESULTS v1 (copy so nay vao bang LaTeX)
def meme_events(bt_df, H=20, th=0.50):
    evs = []
    for sym, g in bt_df.groupby("ticker"):
        g = g.sort_values("date").reset_index(drop=True); c = g["close"].values.astype(float); n = len(g); t = 0
        while t < n - 1:
            w = c[t+1:min(t+1+H, n)]
            if len(w) == 0: break
            if w.max()/c[t]-1 >= th:
                pk = t+1+int(np.argmax(w))
                evs.append({"ticker": sym, "ignition_date": g["date"].iloc[t],
                            "peak_date": g["date"].iloc[pk]}); t = pk+1
            else: t += 1
    return pd.DataFrame(evs)

if len(v1_taken) == 0:
    print("Khong co trade - noi nguong o CFG['L2'].")
else:
    ev = meme_events(bt)
    tk = v1_taken.copy()
    tk["entry_date"] = pd.to_datetime(tk["entry_date"]); tk["exit_date"] = pd.to_datetime(tk["exit_date"])
    r = tk["ret"]; w, ls = tk[r > 0], tk[r <= 0]
    final = tk["equity"].iloc[-1]
    sharpe = r.mean()/r.std(ddof=1) if len(r) > 1 and r.std(ddof=1) > 0 else np.nan
    eqv = tk["equity"].values; pk = np.maximum.accumulate(eqv); dd = ((eqv-pk)/pk).min()
    pf = w["pnl"].sum()/abs(ls["pnl"].sum()) if len(ls) and ls["pnl"].sum() != 0 else float("inf")

    def lab(row):
        e = ev[(ev.ticker == row.ticker) & (ev.ignition_date <= row.exit_date) & (ev.peak_date >= row.entry_date)]
        if len(e) == 0: return "no-meme"
        x = e.sort_values("ignition_date").iloc[0]
        return "early" if row.entry_date < x.ignition_date else ("reactive" if row.entry_date <= x.peak_date else "late")
    tk["timing"] = tk.apply(lab, axis=1)

    print("=" * 62); print("V1 RESULTS"); print("=" * 62)
    print(f"  Trades          : {len(tk)}")
    print(f"  Win rate        : {(r>0).mean():.1%}")
    print(f"  Avg win / loss  : {w['ret'].mean():+.1%} / {ls['ret'].mean():+.1%}")
    print(f"  Expectancy      : {r.mean():+.2%}/trade")
    print(f"  Profit factor   : {pf:.2f}")
    print(f"  Sharpe (trade)  : {sharpe:.2f}")
    print(f"  Max drawdown    : {dd:.1%}")
    print(f"  Final portfolio : ${final:,.0f}  ({final/S['start_capital']-1:+.1%})")
    print("\n  Timing:"); print(tk.groupby("timing").agg(n=("ret","size"),
          win=("ret", lambda x: (x>0).mean()), pnl=("pnl","sum")).round(3).to_string())
    print("\n  Exit reason (cai nao ket thuc lenh):")
    print(tk.groupby("exit_reason").agg(n=("ret","size"), win=("ret", lambda x: (x>0).mean()),
          avg_ret=("ret","mean"), pnl=("pnl","sum")).round(3).to_string())
    print("\n  Per-year:"); print(tk.assign(y=tk.entry_date.dt.year).groupby("y").agg(
          n=("ret","size"), win=("ret", lambda x: (x>0).mean()), pnl=("pnl","sum")).round(3).to_string())
    tk.to_csv(CACHE / "v1_trades.csv", index=False)
    print("\n  Saved: meme_cache/v1_trades.csv")

In [ ]:
# Cell 6b - METRICS DUNG CHUAN: equity mark-to-market HANG NGAY -> Sharpe annualized
# Sharpe = mean(daily_ret)/std(daily_ret)*sqrt(252). KHONG phai mean/std tren tung lenh.
from collections import defaultdict

def daily_equity(taken, bt_df, alloc=None, cap=None):
    """Dung equity MTM theo ngay: moi vi the dong gop shares*(close_t - close_{t-1})."""
    alloc = alloc or S["alloc"]; cap = cap or S["start_capital"]
    b = bt_df.copy(); b["date"] = pd.to_datetime(b["date"]).dt.tz_localize(None)
    px = {s: g.sort_values("date").reset_index(drop=True) for s, g in b.groupby("ticker")}
    acc = defaultdict(float)
    for _, t in taken.iterrows():
        g = px.get(t["ticker"])
        if g is None: continue
        ed = pd.to_datetime(t["entry_date"]).tz_localize(None) if pd.to_datetime(t["entry_date"]).tzinfo else pd.to_datetime(t["entry_date"])
        xd = pd.to_datetime(t["exit_date"]).tz_localize(None) if pd.to_datetime(t["exit_date"]).tzinfo else pd.to_datetime(t["exit_date"])
        ei = g.index[g["date"] == ed]; xi = g.index[g["date"] == xd]
        if len(ei) == 0 or len(xi) == 0: continue
        e, x = int(ei[0]), int(xi[0]); sh = alloc / t["entry_px"]
        cl = g["close"].values.astype(float); dt = g["date"].values
        for j in range(e + 1, x + 1):
            acc[dt[j]] += sh * (cl[j] - cl[j-1])
    cal = pd.DatetimeIndex(sorted(b["date"].unique()))
    daily = pd.Series([acc.get(d.to_datetime64(), 0.0) for d in cal], index=cal)
    return cap + daily.cumsum(), daily

def report_metrics(taken, bt_df):
    eq, daily = daily_equity(taken, bt_df)
    ret = eq.pct_change().dropna()
    yrs = (eq.index[-1] - eq.index[0]).days / 365.25
    cagr = (eq.iloc[-1] / S["start_capital"]) ** (1/yrs) - 1
    vol  = ret.std() * np.sqrt(252)
    shp  = ret.mean() / ret.std() * np.sqrt(252) if ret.std() > 0 else np.nan
    dn   = ret[ret < 0]
    sor  = ret.mean() / dn.std() * np.sqrt(252) if len(dn) and dn.std() > 0 else np.nan
    pk   = eq.cummax(); dd = ((eq - pk) / pk).min()
    r    = taken["ret"]; shp_tr = r.mean() / r.std(ddof=1)
    try:    mo = eq.resample("ME").last().pct_change().dropna()
    except Exception: mo = eq.resample("M").last().pct_change().dropna()

    print("=" * 62); print("METRICS (equity MTM hang ngay - CHUAN)"); print("=" * 62)
    print(f"  CAGR                : {cagr:8.1%}")
    print(f"  Vol annualized      : {vol:8.1%}")
    print(f"  SHARPE annualized   : {shp:8.2f}   <-- con so dung")
    print(f"  Sortino annualized  : {sor:8.2f}   <-- hop hon (phan phoi lech phai)")
    print(f"  Max DD (daily MTM)  : {dd:8.1%}")
    print(f"  Calmar              : {cagr/abs(dd):8.2f}")
    print(f"  Sharpe theo thang   : {mo.mean()/mo.std()*np.sqrt(12):8.2f}  ({len(mo)} thang)")
    print(f"  % ngay co vi the    : {(daily != 0).mean():8.0%}")
    print(f"\\n  [tham chieu] mean/std tung LENH = {shp_tr:.2f} - KHONG phai Sharpe (khong co truc thoi gian)")
    print(f"  std/lenh {r.std(ddof=1):.0%} | skew {r.skew():+.1f} | median {r.median():+.1%} | "
          f"max win {r.max():+.0%}")
    try:    yr = eq.resample("YE").last().pct_change().dropna()
    except Exception: yr = eq.resample("Y").last().pct_change().dropna()
    print("\\n  Return theo nam:")
    for d, x in yr.items(): print(f"    {d.year}: {x:+7.1%}")
    return eq

if len(v1_taken):
    eq_curve = report_metrics(v1_taken, bt)
    try:
        import matplotlib.pyplot as plt
        fig, ax = plt.subplots(1, 2, figsize=(14, 4))
        ax[0].plot(eq_curve.index, eq_curve.values, lw=1.2)
        ax[0].axhline(S["start_capital"], ls="--", c="gray"); ax[0].set_title("Equity MTM hang ngay")
        ax[0].grid(alpha=.3)
        dd_s = (eq_curve - eq_curve.cummax()) / eq_curve.cummax()
        ax[1].fill_between(dd_s.index, dd_s.values, 0, color="crimson", alpha=.5)
        ax[1].set_title("Drawdown"); ax[1].grid(alpha=.3)
        plt.tight_layout(); plt.show()
    except Exception as e:
        print("(skip plot:", e, ")")


In [ ]:
# Cell 7 - (OPTIONAL, MAC DINH TAT) LLM overlay: chi context, KHONG quyet dinh
# comment 7: composite deterministic tinh xong TRUOC, LLM chi phu len top-N.
# comment 6: da co san san pham "summarize" mien phi truoc khi ton token LLM:
#   - StockTwits: moi message CO SAN nhan sentiment Bullish/Bearish -> dem ty le, khong can LLM
#   - ApeWisdom: co san upvotes + rank change -> proxy do quan tam
#   -> LLM chi dung cho NARRATIVE/CATALYST tren <= top_n ten, 1 lan/ngay, cache theo (ticker, ngay).
LLM_CACHE = CACHE / "llm_context.json"

def stocktwits_presummary(sym):
    """Sentiment da duoc gan san boi StockTwits -> MIEN PHI, khong ton token."""
    try:
        r = requests.get(f"https://api.stocktwits.com/api/2/streams/symbol/{sym}.json",
                         headers=CFG["http_headers"], timeout=8)
        if r.status_code != 200: return None
        msgs = r.json().get("messages", [])
        lab = [(m.get("entities", {}).get("sentiment") or {}).get("basic") for m in msgs]
        bull = sum(1 for x in lab if x == "Bullish"); bear = sum(1 for x in lab if x == "Bearish")
        return {"ticker": sym, "n_msg": len(msgs), "bull": bull, "bear": bear,
                "bull_ratio": bull / max(bull + bear, 1)}
    except Exception:
        return None

def llm_context(sym, texts):
    """Chi goi khi CFG['LLM']['enabled']=True. Cache theo (ticker, ngay) -> 1 call/ten/ngay."""
    if not CFG["LLM"]["enabled"]: return None
    key = f"{sym}:{datetime.now():%Y-%m-%d}"
    cache = json.loads(LLM_CACHE.read_text()) if LLM_CACHE.exists() else {}
    if key in cache: return cache[key]
    payload = {"model": os.getenv("DEEPSEEK_MODEL", ""),
               "messages": [
                   {"role": "system", "content":
                    "Summarize retail chatter about ONE ticker. STRICT JSON: "
                    '{"regime":"bull|bear|mixed","narrative":str,"catalyst":str,"confidence":0-1}. '
                    "Do NOT judge whether it is a meme stock."},
                   {"role": "user", "content": f"TICKER {sym}\n{texts[:4000]}"}],
               "temperature": 0.1, "max_tokens": 250,
               "response_format": {"type": "json_object"}}
    try:
        r = requests.post(os.getenv("DEEPSEEK_BASE_URL", "https://openrouter.ai/api/v1") + "/chat/completions",
                          json=payload, timeout=30,
                          headers={"Authorization": f"Bearer {os.getenv('DEEPSEEK_API_KEY','')}",
                                   "Content-Type": "application/json"})
        out = r.json()["choices"][0]["message"]["content"]
        cache[key] = out; LLM_CACHE.write_text(json.dumps(cache))
        return out
    except Exception as e:
        log.warning("LLM fail %s: %s", sym, e); return None

print("LLM overlay loaded | enabled =", CFG["LLM"]["enabled"], "(v1 chay khong can LLM)")

---
# Section 7 - TEARSHEET & ROBUSTNESS (ban present)

In [ ]:
# Cell 7a - TEARSHEET 1: equity (log), rolling Sharpe, monthly heatmap, underwater
import matplotlib.pyplot as plt

BTN = bt.copy(); BTN["date"] = pd.to_datetime(BTN["date"]).dt.tz_localize(None)
PX  = {s: g.sort_values("date").reset_index(drop=True) for s, g in BTN.groupby("ticker")}

eq, daily = daily_equity(v1_taken, bt)
ret  = eq.pct_change().dropna()
yrs  = (eq.index[-1] - eq.index[0]).days / 365.25
cagr = (eq.iloc[-1]/S["start_capital"])**(1/yrs) - 1
shp  = ret.mean()/ret.std()*np.sqrt(252)

fig = plt.figure(figsize=(15, 10))
gs  = fig.add_gridspec(3, 2, height_ratios=[1.25, 1, 1], hspace=.45, wspace=.22)

ax = fig.add_subplot(gs[0, :])
ax.plot(eq.index, eq.values, lw=1.5, color="#1f4e79")
ax.set_yscale("log"); ax.axhline(S["start_capital"], ls="--", c="gray", lw=.8)
ax.set_title(f"Equity (log scale)  |  final ${eq.iloc[-1]:,.0f}   CAGR {cagr:.1%}   Sharpe {shp:.2f}",
             fontsize=12, weight="bold"); ax.grid(alpha=.3)

ax = fig.add_subplot(gs[1, 0])
for win, lbl, col in [(126, "6 thang", "#2e9e5b"), (252, "12 thang", "#c0392b")]:
    rs = ret.rolling(win).mean()/ret.rolling(win).std()*np.sqrt(252)
    ax.plot(rs.index, rs.values, lw=1.2, label=lbl, color=col)
ax.axhline(0, c="k", lw=.8); ax.axhline(1, c="gray", ls=":", lw=.8)
ax.axhline(shp, c="#1f4e79", ls="--", lw=1, label=f"full {shp:.2f}")
ax.set_title("Rolling Sharpe (annualized)"); ax.legend(fontsize=8); ax.grid(alpha=.3)

ax = fig.add_subplot(gs[1, 1])
dd = (eq - eq.cummax())/eq.cummax()
ax.fill_between(dd.index, dd.values*100, 0, color="crimson", alpha=.5)
ax.set_title(f"Underwater (max DD {dd.min():.1%})"); ax.set_ylabel("%"); ax.grid(alpha=.3)

ax = fig.add_subplot(gs[2, :])
try:    mo = eq.resample("ME").last().pct_change().dropna()
except Exception: mo = eq.resample("M").last().pct_change().dropna()
piv = pd.DataFrame({"y": mo.index.year, "m": mo.index.month, "r": mo.values}).pivot(index="y", columns="m", values="r")
im = ax.imshow(piv.values*100, cmap="RdYlGn", aspect="auto", vmin=-30, vmax=30)
ax.set_xticks(range(len(piv.columns))); ax.set_xticklabels(piv.columns)
ax.set_yticks(range(len(piv.index)));  ax.set_yticklabels(piv.index)
for i in range(piv.shape[0]):
    for j in range(piv.shape[1]):
        v = piv.values[i, j]
        if not np.isnan(v): ax.text(j, i, f"{v*100:.0f}", ha="center", va="center", fontsize=7)
ax.set_title("Monthly returns (%)"); plt.colorbar(im, ax=ax, fraction=.02)
plt.suptitle("MEME SCANNER v1 - PERFORMANCE TEARSHEET", fontsize=13, weight="bold", y=.985)
plt.show()


In [ ]:
# Cell 7b - TEARSHEET 2: thong ke return + phan phoi
r = v1_taken["ret"]; dr = ret
def stats(x, lbl):
    return {"series": lbl, "n": len(x), "mean": x.mean(), "median": x.median(), "std": x.std(),
            "skew": x.skew(), "kurt": x.kurtosis(), "min": x.min(), "max": x.max(),
            "VaR95": np.percentile(x, 5), "CVaR95": x[x <= np.percentile(x, 5)].mean()}
st = pd.DataFrame([stats(r, "per-trade"), stats(dr, "daily")]).set_index("series")
print("=" * 78); print("THONG KE RETURN"); print("=" * 78)
print(st.applymap(lambda v: f"{v:,.4f}" if isinstance(v, float) else v).to_string())

w, l = r[r > 0], r[r <= 0]
print(f"\n  Win rate {len(w)/len(r):.1%} | avg win {w.mean():+.1%} | avg loss {l.mean():+.1%} "
      f"| payoff {abs(w.mean()/l.mean()):.2f}")
print(f"  Top 10 lenh dong gop {v1_taken.nlargest(10,'pnl')['pnl'].sum()/v1_taken['pnl'].sum():.0%} tong P&L")
print(f"  Bo 10 winner lon nhat -> P&L con ${v1_taken['pnl'].sum()-v1_taken.nlargest(10,'pnl')['pnl'].sum():,.0f}")

fig, ax = plt.subplots(2, 2, figsize=(14, 7.5)); ax = ax.ravel()
ax[0].hist(np.clip(r, -.5, 2.0), bins=60, color="#3b6ea5", edgecolor="w")
ax[0].axvline(0, c="k", lw=.8); ax[0].axvline(r.mean(), c="crimson", ls="--", label=f"mean {r.mean():+.1%}")
ax[0].set_title(f"Trade returns (clip 200%) | skew {r.skew():+.1f}"); ax[0].legend(fontsize=8); ax[0].grid(alpha=.3)

ax[1].hist(dr, bins=80, density=True, color="#7aa6c2", edgecolor="w", label="thuc te")
xs = np.linspace(dr.min(), dr.max(), 200)
ax[1].plot(xs, (1/(dr.std()*np.sqrt(2*np.pi)))*np.exp(-.5*((xs-dr.mean())/dr.std())**2),
           "r--", lw=1.2, label="normal")
ax[1].set_title("Daily returns vs normal"); ax[1].legend(fontsize=8); ax[1].grid(alpha=.3)

try:
    from scipy import stats as sps
    sps.probplot(dr, dist="norm", plot=ax[2]); ax[2].set_title("QQ plot (daily)")
except Exception:
    ax[2].text(.5, .5, "scipy khong co", ha="center")
ax[2].grid(alpha=.3)

srt = v1_taken.sort_values("pnl", ascending=False)["pnl"].cumsum()/v1_taken["pnl"].sum()
ax[3].plot(range(1, len(srt)+1), srt.values, lw=1.5, color="#8e44ad")
ax[3].axhline(1, ls="--", c="gray"); ax[3].set_xlabel("so lenh (xep theo P&L giam dan)")
ax[3].set_title("Duong tap trung P&L (cang doc = cang phu thuoc it lenh)"); ax[3].grid(alpha=.3)
plt.tight_layout(); plt.show()


In [ ]:
# Cell 7c - VE TUNG TRADE: top 10 lai / top 10 lo
def _idx_of(sym, d):
    g = PX.get(sym)
    if g is None: return None, None
    d = pd.to_datetime(d)
    if d.tzinfo is not None: d = d.tz_localize(None)
    i = g.index[g["date"] == d]
    return (g, int(i[0])) if len(i) else (g, None)

def plot_trade(row, ax, pad=30):
    g, ei = _idx_of(row["ticker"], row["entry_date"])
    _, xi = _idx_of(row["ticker"], row["exit_date"])
    if g is None or ei is None or xi is None:
        ax.text(.5, .5, f"{row['ticker']}: no data", ha="center"); return
    a, b = max(0, ei-pad), min(len(g)-1, xi+pad)
    s = g.iloc[a:b+1]
    ax.plot(s["date"], s["close"], lw=1.1, color="#333")
    ma20 = g["close"].rolling(20).mean().iloc[a:b+1]
    ax.plot(s["date"], ma20, lw=.9, ls="--", color="#888", label="MA20")
    ax.axvspan(g["date"].iloc[ei], g["date"].iloc[xi], color="#ffe08a", alpha=.35)
    ax.scatter([g["date"].iloc[ei]], [row["entry_px"]], marker="^", s=90, color="green", zorder=5)
    ax.scatter([g["date"].iloc[xi]], [row["exit_px"]], marker="v", s=90, color="red", zorder=5)
    ax.set_title(f"{row['ticker']}  {row['ret']:+.0%}  ({row['days']}d, {row['exit_reason']})", fontsize=9)
    ax.tick_params(labelsize=7); ax.grid(alpha=.25)
    for lb in ax.get_xticklabels(): lb.set_rotation(30)

for title, sub in [("TOP 10 LENH LAI", v1_taken.nlargest(10, "ret")),
                   ("TOP 10 LENH LO",  v1_taken.nsmallest(10, "ret"))]:
    fig, axes = plt.subplots(2, 5, figsize=(17, 6.5))
    for ax, (_, row) in zip(axes.ravel(), sub.iterrows()): plot_trade(row, ax)
    plt.suptitle(title + "   (^ vao, v ra, vung vang = thoi gian giu)", fontsize=12, weight="bold")
    plt.tight_layout(); plt.show()

print("TOP 10 LAI:");  print(v1_taken.nlargest(10, "ret")[
    ["ticker","entry_date","entry_px","exit_date","exit_px","ret","days","rvol_in","exit_reason"]].to_string(index=False))
print("\nTOP 10 LO:"); print(v1_taken.nsmallest(10, "ret")[
    ["ticker","entry_date","entry_px","exit_date","exit_px","ret","days","rvol_in","exit_reason"]].to_string(index=False))


## Robustness - 5 test truoc khi ket luan

In [ ]:
# Cell 7d - ROBUSTNESS SUITE: 5 test de biet co overfit khong
def ann_sharpe(taken):
    if len(taken) < 5: return np.nan
    e, _ = daily_equity(taken, bt); rr = e.pct_change().dropna()
    return rr.mean()/rr.std()*np.sqrt(252) if rr.std() > 0 else np.nan

print("=" * 78); print("TEST 1 - PARAMETER SENSITIVITY (be mat phang hay dinh nhon?)"); print("=" * 78)
grid = pd.DataFrame(index=[1.5,1.75,2.0,2.5,3.0,4.0,5.0], columns=[0.20,0.30,0.50,0.70,1.00], dtype=float)
bak_r, bak_e = CFG["L2"]["rvol_min"], CFG["L2"]["max_ext_ma20"]
for rv in grid.index:
    for ex in grid.columns:
        CFG["L2"]["rvol_min"], CFG["L2"]["max_ext_ma20"] = rv, ex
        t = backtest_v1(bt); tt = portfolio(t) if len(t) else pd.DataFrame()
        grid.loc[rv, ex] = ann_sharpe(tt) if len(tt) else np.nan
CFG["L2"]["rvol_min"], CFG["L2"]["max_ext_ma20"] = bak_r, bak_e
print("Sharpe annualized theo (rvol_min = hang) x (max_ext_ma20 = cot):")
print(grid.round(2).to_string())
flat = grid.values.flatten(); flat = flat[~np.isnan(flat)]
print(f"\n  best {np.nanmax(flat):.2f} | median {np.median(flat):.2f} | ty le {np.nanmax(flat)/max(np.median(flat),.01):.2f}x")
print("  -> gan 1.5x = CAO NGUYEN (ben). > 3x = DINH NHON (overfit).")
try:
    fig, ax = plt.subplots(figsize=(7, 4))
    im = ax.imshow(grid.values.astype(float), cmap="RdYlGn", aspect="auto")
    ax.set_xticks(range(len(grid.columns))); ax.set_xticklabels(grid.columns)
    ax.set_yticks(range(len(grid.index)));  ax.set_yticklabels(grid.index)
    ax.set_xlabel("max_ext_ma20"); ax.set_ylabel("rvol_min"); ax.set_title("Sharpe surface")
    for i in range(grid.shape[0]):
        for j in range(grid.shape[1]):
            ax.text(j, i, f"{grid.values[i,j]:.2f}", ha="center", va="center", fontsize=8)
    plt.colorbar(im); plt.tight_layout(); plt.show()
except Exception as e: print("(skip heatmap)", e)

print("\n" + "=" * 78); print("TEST 2 - WALK-FORWARD (3 fold theo thoi gian)"); print("=" * 78)
tkw = v1_taken.copy(); tkw["entry_date"] = pd.to_datetime(tkw["entry_date"])
qs = tkw["entry_date"].quantile([0, 1/3, 2/3, 1.0]).tolist()
for k in range(3):
    seg = tkw[(tkw["entry_date"] >= qs[k]) & (tkw["entry_date"] <= qs[k+1])]
    if len(seg):
        print(f"  Fold {k+1} [{qs[k]:%Y-%m} -> {qs[k+1]:%Y-%m}]: n={len(seg):3d} "
              f"win {(seg['ret']>0).mean():.0%} exp {seg['ret'].mean():+.2%} pnl ${seg['pnl'].sum():>10,.0f}")
print("  -> Fold cuoi AM trong khi fold dau duong = edge dang mat hieu luc.")

print("\n" + "=" * 78); print("TEST 3 - MONTE CARLO (bootstrap thu tu lenh, 1000 lan)"); print("=" * 78)
rng = np.random.default_rng(7); rr = v1_taken["ret"].values; finals, dds = [], []
for _ in range(1000):
    s = rng.choice(rr, size=len(rr), replace=True)
    eqp = S["start_capital"] + np.cumsum(s*S["alloc"])
    pk = np.maximum.accumulate(eqp); finals.append(eqp[-1]); dds.append(((eqp-pk)/pk).min())
finals = np.array(finals); dds = np.array(dds)
print(f"  Final equity: p5 ${np.percentile(finals,5):,.0f} | median ${np.percentile(finals,50):,.0f} "
      f"| p95 ${np.percentile(finals,95):,.0f}")
print(f"  P(co lai) = {(finals > S['start_capital']).mean():.1%}   |  Max DD median {np.median(dds):.1%}, p95 {np.percentile(dds,5):.1%}")

print("\n" + "=" * 78); print("TEST 4 - NULL TEST (entry NGAU NHIEN cung ten/cung ky/cung so ngay giu)"); print("=" * 78)
hold = v1_taken["days"].values; syms = v1_taken["ticker"].unique().tolist(); null = []
for _ in range(200):
    acc = []
    for _ in range(len(v1_taken)):
        sy = syms[rng.integers(len(syms))]; g = PX.get(sy)
        if g is None or len(g) < 80: continue
        hd = int(hold[rng.integers(len(hold))]); i = int(rng.integers(21, len(g)-hd-1))
        c = g["close"].values.astype(float); acc.append(c[i+hd]/c[i]-1)
    if acc: null.append(np.mean(acc))
null = np.array(null); act = v1_taken["ret"].mean()
print(f"  Entry ngau nhien : mean ret {null.mean():+.2%}  (sd {null.std():.2%})")
print(f"  Entry cua CHIEN LUOC: {act:+.2%}")
print(f"  z-score = {(act-null.mean())/max(null.std(),1e-9):.2f}  |  p ~ {(null >= act).mean():.3f}")
print("  -> p < 0.05 = tin hieu THAT SU tot hon vao ngau nhien.")

print("\n" + "=" * 78); print("TEST 5 - LEAVE-ONE-TICKER-OUT (co phu thuoc 1 ma khong?)"); print("=" * 78)
tot = v1_taken["pnl"].sum()
by = v1_taken.groupby("ticker")["pnl"].sum().sort_values(ascending=False)
print("  5 ma dong gop nhieu nhat:")
for t, v in by.head(5).items(): print(f"    {t:6s} ${v:>10,.0f}  ({v/tot:5.1%})  -> bo ra con ${tot-v:,.0f}")
print(f"  Bo 3 ma tot nhat -> P&L con ${tot-by.head(3).sum():,.0f} ({(tot-by.head(3).sum())/tot:.0%} con lai)")


## Chot chien luoc

In [ ]:
# Cell 7e - CHOT CHIEN LUOC & CHAY BAN CUOI
FINAL = {
    # rvol_min=2.0: KHONG chon vi Sharpe cao nhat, ma vi NHIEU LENH hon (866 vs 577)
# -> thong ke vung hon + P&L bot tap trung (top3 ticker 60% vs 79%). Hang xom 1.75/2.5 van OK.
    "L2":   {"rvol_min": 2.0, "max_ext_ma20": 0.50, "require_above_ma20": True,
             "mom5_min": 0.05, "require_above_vwap": True, "require_follow_through": True},
    "EXIT": {"hard_stop": 0.08, "use_no_meme": False, "atr_mult": 3.0, "ma_exit": 10,
             "max_hold": 60, "thesis_monitor": True},
    "L1":   {"si_missing": "allow"},
}
for k, v in FINAL.items(): CFG[k].update(v)

final_trades = backtest_v1(bt)
final_taken  = portfolio(final_trades)
e, _ = daily_equity(final_taken, bt); rr = e.pct_change().dropna()
yrs = (e.index[-1]-e.index[0]).days/365.25
card = {
    "Trades": f"{len(final_taken)}",
    "Win rate": f"{(final_taken['ret']>0).mean():.1%}",
    "Expectancy/trade": f"{final_taken['ret'].mean():+.2%}",
    "CAGR": f"{(e.iloc[-1]/S['start_capital'])**(1/yrs)-1:.1%}",
    "Sharpe (ann.)": f"{rr.mean()/rr.std()*np.sqrt(252):.2f}",
    "Sortino (ann.)": f"{rr.mean()/rr[rr<0].std()*np.sqrt(252):.2f}",
    "Max DD": f"{((e-e.cummax())/e.cummax()).min():.1%}",
    "Final equity": f"${e.iloc[-1]:,.0f}",
}
print("=" * 56); print("STRATEGY CARD - MEME SCANNER v1 (FINAL)"); print("=" * 56)
for k, v in card.items(): print(f"  {k:20s}: {str(v):>14s}")
L2 = CFG["L2"]; EX = CFG["EXIT"]
print(f"\n  ENTRY : RVOL>={L2['rvol_min']} & close>VWAP & close>prev & close>MA20 "
      f"& mom5>{L2['mom5_min']:.0%} & ext<=+{L2['max_ext_ma20']:.0%} & short-fuel")
print(f"  EXIT  : -{EX['hard_stop']:.0%} hard stop | {EX['atr_mult']}xATR trail | "
      f"gay MA{EX['ma_exit']} sau khi lai | thesis mat | max {EX['max_hold']} phien" + (f" | no-meme {EX['no_meme_days']}d" if EX["use_no_meme"] else ""))
final_taken.to_csv(CACHE / "v1_final_trades.csv", index=False)
print("\n  Saved: meme_cache/v1_final_trades.csv")


In [ ]:
# Cell 7f - SCORECARD: chot lai 5 test, cai nao PASS cai nao RED FLAG
tot = final_taken["pnl"].sum()
top10 = final_taken.nlargest(10, "pnl")["pnl"].sum()/tot
by = final_taken.groupby("ticker")["pnl"].sum().sort_values(ascending=False)
top3t = by.head(3).sum()/tot
e2, _ = daily_equity(final_taken, bt)
try:    yr = e2.resample("YE").last().pct_change().dropna()
except Exception: yr = e2.resample("Y").last().pct_change().dropna()
neg_recent = [f"{d.year}:{v:+.0%}" for d, v in yr.items()][-2:]

score = [
 ("1. Parameter surface", "PASS",
  "Sharpe 0.83-0.91 tren vung rvol 1.75-3.0 x ext 0.3-1.0 -> CAO NGUYEN, khong phai dinh nhon"),
 ("2. Walk-forward",      "PASS*",
  "3 fold deu duong. *NHUNG fold chia theo SO LENH, khong theo thoi gian -> che mat 2025-26 am"),
 ("3. Monte Carlo",       "PASS",
  "1000 bootstrap: P(lai)=98.6%, p5 van $178k -> khong phai an may thu tu lenh"),
 ("4. Null test",         "PASS",
  "strategy +6.09% vs entry ngau nhien +0.91% (sd 2.02%), z=2.56, p=0.025 -> tin hieu THAT"),
 ("5. Concentration",     "RED FLAG",
  f"top10 lenh = {top10:.0%} tong P&L; top3 ticker = {top3t:.0%} -> 500+ lenh con lai gop NET AM"),
 ("6. Regime gan day",    "RED FLAG",
  f"nam gan nhat: {', '.join(neg_recent)} -> edge dang yeu di o regime hien tai"),
]
print("=" * 92); print("SCORECARD - CO OVERFIT KHONG?"); print("=" * 92)
for name, verd, note in score:
    print(f"  {name:24s} [{verd:9s}] {note}")
print("\n" + "=" * 92); print("KET LUAN"); print("=" * 92)
print("""  Tham so VUNG (cao nguyen) va tin hieu THAT (thang entry ngau nhien, p=0.025).
  KHONG phai overfit tham so. NHUNG day la chien luoc DUOI DAY (fat tail):
  toan bo loi nhuan den tu ~10 lenh / ~3 ma. Nghia la:
    - phai vao DU nhieu lenh, khong duoc bo lo cu lon
    - phai song sot qua chuoi lo dai (maxDD ~45%)
    - dung danh gia bang win rate (33%) - danh gia bang Sortino (1.5) va expectancy
  Rui ro that: 2025-2026 am -> can forward paper truoc khi bo tien that.""")


---
# Section 8 - AUDIT BIAS, EARLY EXIT, INTRADAY

In [ ]:
# Cell 8a - AUDIT EXECUTION: "action tai t, data chi toi t-1"
# 3 muc do thuc te. Ban goc quyet dinh VA vao lenh cung tai close[i] -> hoi lac quan.
def bt_exec(mode, rvol_th=2.0, ext_cap=0.50, early=None):
    """mode: close_close | next_open | next_open_stop
       early: None hoac ('mfe', n_days, thresh) - thoat som khi khong tien trien"""
    tr=[]
    for sym, g in PX.items():
        if len(g) < 60: continue
        c=g["close"].values.astype(float); v=g["vol"].values.astype(float)
        h=g["high"].values.astype(float); l=g["low"].values.astype(float)
        o=g["open"].values.astype(float) if "open" in g else c
        n=len(g); atr=_atr(h,l,c,14); i=21
        while i < n-2:
            base=v[i-20:i].mean(); rv=v[i]/base if base>0 else 0
            ma20=c[i-20:i].mean(); ext=c[i]/ma20-1 if ma20>0 else 0
            typ=(h[i]+l[i]+c[i])/3
            sig=(rv>=rvol_th and c[i]>typ and c[i]>c[i-1] and ext<=ext_cap and c[i]>ma20
                 and c[i]/c[i-5]-1>0.05 and _si_ok(sym,g["date"].iloc[i]))
            if sig:
                ei,entry = (i,c[i]) if mode=="close_close" else (i+1,o[i+1])
                if entry<=0: i+=1; continue
                peak=entry; xi=min(ei+60,n-1); ex=None; rs="max_hold"
                for j in range(ei+1,xi+1):
                    stop=entry*0.92; k=j-ei
                    if mode=="next_open_stop":
                        if o[j]<=stop: xi,ex,rs=j,o[j],"stop_gap"; break
                        if l[j]<=stop: xi,ex,rs=j,stop,"stop"; break
                    elif c[j]<=stop: xi,ex,rs=j,c[j],"stop_close"; break
                    if early and early[0]=="mfe" and k>=early[1] and (c[ei+1:j+1].max()/entry-1)<early[2]:
                        xi,ex,rs=j,c[j],"early_exit"; break
                    peak=max(peak,c[j])
                    if c[j]<=peak-3.0*(atr[j] if not np.isnan(atr[j]) else 0): xi,ex,rs=j,c[j],"atr_trail"; break
                    if c[j]>entry and c[j]<c[max(0,j-9):j+1].mean(): xi,ex,rs=j,c[j],"ma_break"; break
                if ex is None: ex=c[xi]
                tr.append({"ticker":sym,"ei":ei,"xi":xi,"entry_date":g["date"].iloc[ei],
                           "exit_date":g["date"].iloc[xi],"entry_px":entry,"exit_px":ex,
                           "ret":ex/entry-1,"days":xi-ei,"exit_reason":rs})
                i=xi+1; continue
            i+=1
    td=pd.DataFrame(tr)
    if len(td)==0: return td
    td=td.sort_values("entry_date").reset_index(drop=True)
    rows,opens=[],[]
    for _,t in td.iterrows():
        opens=[d for d in opens if d>t["entry_date"]]
        if len(opens)>=S["max_positions"]: continue
        opens.append(t["exit_date"]); rows.append({**t.to_dict(),"pnl":S["alloc"]*t["ret"]})
    return pd.DataFrame(rows)

def quick(tk, lbl):
    if len(tk)==0: print(f"{lbl:52s} 0 trades"); return
    e,_=daily_equity(tk,bt); r=e.pct_change().dropna()
    yrs=(e.index[-1]-e.index[0]).days/365.25
    print(f"{lbl:52s} n={len(tk):4d} win {(tk.ret>0).mean():5.1%} exp {tk.ret.mean():+6.2%} "
          f"CAGR {(e.iloc[-1]/S['start_capital'])**(1/yrs)-1:6.1%} Sharpe {r.mean()/r.std()*np.sqrt(252):5.2f} "
          f"final ${e.iloc[-1]:,.0f}")

print("="*104); print("AUDIT: co dung 'data toi t-1' khong?"); print("="*104)
quick(bt_exec("close_close"),    "A. quyet dinh & VAO tai close[i]        (ban goc)")
quick(bt_exec("next_open"),      "B. quyet dinh close[i] -> VAO open[i+1]  (dung chuan)")
quick(bt_exec("next_open_stop"), "C. B + stop fill theo LOW/GAP            (thuc te nhat)")
print("""
  A -> B: chenh lech = muc do LAC QUAN cua viec gia dinh khop dung gia close ma minh dung de quyet dinh.
  C la ban DUNG NHAT: vao lenh o mo cua phien sau, stop khop trong phien (hoac gap qua stop).
  Cac phan con lai KHONG co look-ahead: MA/RVOL chi dung bar <= i, short-interest tru 12 ngay cong bo,
  vong exit quet bar tuong lai = trade DANG dien tien (khong dung tuong lai de QUYET DINH vao).""")


In [ ]:
# Cell 8b - TEST BIAS UNIVERSE (quan trong nhat): cung LUAT, ro MEME vs ro NGAU NHIEN
NEU = load_df(CACHE / "neutral_universe.parquet")
if NEU is None:
    print("Chua co neutral_universe.parquet - xem script fetch trong bao cao.")
else:
    def run_on(df, vol_min=0.0, px_max=1e9):
        P={s:g.sort_values("date").reset_index(drop=True) for s,g in df.groupby("ticker")}
        tr=[]
        for sym,g in P.items():
            if len(g)<60: continue
            c=g["close"].values.astype(float); v=g["vol"].values.astype(float)
            h=g["high"].values.astype(float); l=g["low"].values.astype(float)
            o=g["open"].values.astype(float); n=len(g); atr=_atr(h,l,c,14); i=21
            while i<n-2:
                base=v[i-20:i].mean(); rv=v[i]/base if base>0 else 0
                ma20=c[i-20:i].mean(); ext=c[i]/ma20-1 if ma20>0 else 0
                typ=(h[i]+l[i]+c[i])/3
                atrp=atr[i]/c[i] if not np.isnan(atr[i]) and c[i]>0 else 0
                if (rv>=2.0 and c[i]>typ and c[i]>c[i-1] and ext<=0.50 and c[i]>ma20
                    and c[i]/c[i-5]-1>0.05 and atrp>=vol_min and c[i]<=px_max):
                    ei=i+1; entry=o[i+1]
                    if entry<=0: i+=1; continue
                    peak=entry; xi=min(ei+60,n-1); ex=None
                    for j in range(ei+1,xi+1):
                        stop=entry*0.92
                        if o[j]<=stop: xi,ex=j,o[j]; break
                        if l[j]<=stop: xi,ex=j,stop; break
                        peak=max(peak,c[j])
                        if c[j]<=peak-3.0*(atr[j] if not np.isnan(atr[j]) else 0): xi,ex=j,c[j]; break
                        if c[j]>entry and c[j]<c[max(0,j-9):j+1].mean(): xi,ex=j,c[j]; break
                    if ex is None: ex=c[xi]
                    tr.append({"ticker":sym,"ret":ex/entry-1,"entry_date":g["date"].iloc[ei],
                               "exit_date":g["date"].iloc[xi]})
                    i=xi+1; continue
                i+=1
        t=pd.DataFrame(tr)
        return t.assign(pnl=S["alloc"]*t["ret"]) if len(t) else t
    mm = bt.copy(); mm["date"]=pd.to_datetime(mm["date"]).dt.tz_localize(None)
    print("="*96); print("CUNG LUAT ENTRY/EXIT - chi khac RO CO PHIEU"); print("="*96)
    print(f"{'universe':46s} {'n':>5s} {'win':>7s} {'exp/trade':>10s} {'P&L':>12s}")
    for lbl,df,vm,pm in [("MEME (ro chon bang HINDSIGHT)",     mm, 0.0,  1e9),
                         ("NGAU NHIEN 113 ma - khong loc",     NEU,0.0,  1e9),
                         ("NGAU NHIEN + vola>=6%/ngay (ex-ante)",NEU,0.06,1e9),
                         ("NGAU NHIEN + vola>=6% & gia<=$20",  NEU,0.06, 20.0)]:
        t=run_on(df,vm,pm)
        if len(t)<5: print(f"{lbl:46s} {len(t):5d}  qua it"); continue
        print(f"{lbl:46s} {len(t):5d} {(t.ret>0).mean():7.1%} {t.ret.mean():+10.2%} {t.pnl.sum():12,.0f}")
    print("""
  DOC KET QUA: neu chi ro MEME co lai con moi ro ngau nhien (ke ca da loc vola ex-ante) deu LO
  => luat entry KHONG co edge doc lap; P&L den tu viec DANH SACH TEN da duoc chon bang hindsight.
  => Backtest nay chi kiem chung duoc TIMING trong mot ro tot cho san, KHONG kiem chung duoc
     bo phan quan trong nhat: LAYER-1 co tu tim ra ten tot khong.""")


In [ ]:
# Cell 8c - EARLY EXIT: thoat som hon -8% ma KHONG cat mat winner
print("="*104); print("EARLY EXIT - tim rule cat lo som ma giu nguyen winner"); print("="*104)
base_ex = bt_exec("next_open_stop")
bw = (base_ex.ret>=0.5).sum(); btot = base_ex.pnl.sum()
print(f"{'rule':36s} {'n':>5s} {'win':>6s} {'exp':>7s} {'avgLoss':>8s} {'P&L':>11s} {'#win>50%':>9s} {'delta':>10s}")
print(f"{'BASE (chi -8% hard stop)':36s} {len(base_ex):5d} {(base_ex.ret>0).mean():6.1%} "
      f"{base_ex.ret.mean():+7.2%} {base_ex[base_ex.ret<=0].ret.mean():8.1%} {btot:11,.0f} {bw:9d} {'-':>10s}")
for lbl,e in [("MFE<3% sau 2 ngay  <-- TOT NHAT", ("mfe",2,0.03)),
              ("MFE<5% sau 3 ngay",               ("mfe",3,0.05)),
              ("MFE<10% sau 3 ngay (rule cu)",    ("mfe",3,0.10))]:
    t=bt_exec("next_open_stop", early=e)
    if len(t)==0: continue
    w=(t.ret>=0.5).sum()
    print(f"{lbl:36s} {len(t):5d} {(t.ret>0).mean():6.1%} {t.ret.mean():+7.2%} "
          f"{t[t.ret<=0].ret.mean():8.1%} {t.pnl.sum():11,.0f} {w:9d} {t.pnl.sum()-btot:+10,.0f}")
print("""
  '#win>50%' = so lenh lai tren 50%. Rule tot = P&L TANG ma cot nay KHONG tut.
  MFE<3%/2 ngay thang vi no chi cat nhung lenh CHET HAN (chua bao gio nhuc nhich 3%),
  trong khi rule cu (MFE<10%/3 ngay) cat ca nhung lenh dang tich luy -> lo tien.""")


## Intraday 15 phut (can bt_15min.parquet)

In [ ]:
# Cell 8d - INTRADAY 15 PHUT: cung luat nhung thuc thi tren bar 15p (so voi daily)
M15 = load_df(CACHE / "bt_15min.parquet")
if M15 is None:
    print("Chua co bt_15min.parquet. Chay script fetch (xem bao cao) roi quay lai.")
else:
    M15 = M15.copy()
    M15["ts"] = pd.to_datetime(M15["ts"], utc=True).dt.tz_convert("America/New_York")
    M15 = M15[(M15.ts.dt.time >= pd.Timestamp("09:30").time()) &
              (M15.ts.dt.time <  pd.Timestamp("16:00").time())]      # chi gio giao dich chinh
    M15["day"] = M15.ts.dt.normalize().dt.tz_localize(None)
    print(f"15-min: {len(M15):,} bars | {M15.ticker.nunique()} ma | {M15.day.min().date()} -> {M15.day.max().date()}")

    DLY = {s: g.sort_values("date").reset_index(drop=True) for s, g in
           bt.assign(date=lambda d: pd.to_datetime(d["date"]).dt.tz_localize(None)).groupby("ticker")}

    def backtest_15(rvol_th=2.0, ext_cap=0.50, stop=0.08, early=("mfe",2,0.03), max_hold_d=60):
        """Entry danh gia MOI BAR 15p, vao o OPEN bar ke tiep. Context daily lay tu ngay HOM TRUOC."""
        tr=[]
        for sym, m in M15.groupby("ticker"):
            d = DLY.get(sym)
            if d is None or len(d) < 40: continue
            dv = {r.date: (i, r.close) for i, r in enumerate(d.itertuples())}
            dc = d["close"].values.astype(float)
            dh = d["high"].values.astype(float); dl = d["low"].values.astype(float)
            datr = _atr(dh, dl, dc, 14)
            m = m.sort_values("ts").reset_index(drop=True)
            m["slot"] = m.groupby("day").cumcount()
            m["cum_v"] = m.groupby("day")["vol"].cumsum()
            m["cum_pv"] = m.groupby("day").apply(lambda x: (x["close"]*x["vol"]).cumsum()).reset_index(level=0, drop=True)
            m["vwap"] = m["cum_pv"] / m["cum_v"].replace(0, np.nan)
            base = m.groupby("slot")["cum_v"].transform(lambda x: x.shift(1).rolling(20, min_periods=5).mean())
            m["rvol_tod"] = m["cum_v"] / base
            ts=m["ts"].values; op=m["open"].values.astype(float); hi=m["high"].values.astype(float)
            lo=m["low"].values.astype(float); cl=m["close"].values.astype(float)
            vw=m["vwap"].values; rv=m["rvol_tod"].values; dys=m["day"].values
            N=len(m); i=1
            while i < N-1:
                di = dv.get(pd.Timestamp(dys[i]))
                if di is None or di[0] < 21: i+=1; continue
                k = di[0]                                   # index ngay hom nay trong daily
                ma20 = dc[k-20:k].mean()                    # DUNG bar <= hom qua -> khong look-ahead
                prev_close = dc[k-1]; mom5 = dc[k-1]/dc[k-6]-1 if k>=6 else 0
                if (rv[i]>=rvol_th and cl[i]>vw[i] and cl[i]>prev_close and ma20>0
                        and cl[i]/ma20-1 <= ext_cap and cl[i]>ma20 and mom5>0.05):
                    entry=op[i+1]; ei=i+1
                    if entry<=0: i+=1; continue
                    peak=entry; ex=None; rs="max_hold"; xi=N-1
                    last_day = pd.Timestamp(dys[ei]) + pd.Timedelta(days=max_hold_d*1.5)
                    for j in range(ei+1, N):
                        if pd.Timestamp(dys[j]) > last_day: xi,ex,rs=j,cl[j],"max_hold"; break
                        sp = entry*(1-stop)
                        if op[j] <= sp: xi,ex,rs=j,op[j],"stop_gap"; break
                        if lo[j] <= sp: xi,ex,rs=j,sp,"stop"; break
                        nd = (pd.Timestamp(dys[j]) - pd.Timestamp(dys[ei])).days
                        if early and nd>=early[1] and (cl[ei+1:j+1].max()/entry-1) < early[2]:
                            xi,ex,rs=j,cl[j],"early_exit"; break
                        peak=max(peak, cl[j])
                        kk = dv.get(pd.Timestamp(dys[j]))
                        a = datr[kk[0]] if kk and not np.isnan(datr[kk[0]]) else 0
                        if cl[j] <= peak - 3.0*a: xi,ex,rs=j,cl[j],"atr_trail"; break
                    if ex is None: ex=cl[min(xi,N-1)]
                    tr.append({"ticker":sym,"entry_ts":ts[ei],"exit_ts":ts[min(xi,N-1)],
                               "entry_px":entry,"exit_px":ex,"ret":ex/entry-1,"exit_reason":rs,
                               "entry_date":pd.Timestamp(dys[ei]),"exit_date":pd.Timestamp(dys[min(xi,N-1)])})
                    i=min(xi,N-1)+1; continue
                i+=1
        t=pd.DataFrame(tr)
        if len(t)==0: return t
        t=t.sort_values("entry_ts").reset_index(drop=True)
        rows,opens=[],[]
        for _,x in t.iterrows():
            opens=[o for o in opens if o > x["entry_ts"]]
            if len(opens)>=S["max_positions"]: continue
            opens.append(x["exit_ts"]); rows.append({**x.to_dict(),"pnl":S["alloc"]*x["ret"]})
        return pd.DataFrame(rows)

    intraday = backtest_15()
    print(f"\nINTRADAY 15p: {len(intraday)} trades")
    if len(intraday):
        daily_v = bt_exec("next_open_stop", early=("mfe",2,0.03))
        print("\n" + "="*92); print("DAILY vs INTRADAY (cung luat, khac tan suat thuc thi)"); print("="*92)
        print(f"{'':22s} {'n':>5s} {'win':>7s} {'exp':>8s} {'avgWin':>8s} {'avgLoss':>8s} {'P&L':>12s}")
        for lbl,t in [("DAILY (close->open)",daily_v),("INTRADAY 15 phut",intraday)]:
            w=t[t.ret>0]; l=t[t.ret<=0]
            print(f"{lbl:22s} {len(t):5d} {(t.ret>0).mean():7.1%} {t.ret.mean():+8.2%} "
                  f"{w.ret.mean():+8.1%} {l.ret.mean():+8.1%} {t.pnl.sum():12,.0f}")
        print("\nExit reason (intraday):")
        print(intraday.groupby("exit_reason").agg(n=("ret","size"),win=("ret",lambda x:(x>0).mean()),
              avg=("ret","mean"),pnl=("pnl","sum")).round(3).to_string())
        intraday.to_csv(CACHE/"intraday_trades.csv", index=False)


---
# Section 9 - Point-in-time universe selector (sua bias)

In [ ]:
# Cell 9 - POINT-IN-TIME UNIVERSE SELECTOR (chon ro tai t bang du lieu <= t-1)
# Sua dung cai bias o Cell 8b: KHONG dung danh sach ten co dinh chon bang hindsight nua,
# ma moi ngay CHAM DIEM lai toan bo pool bang dac tinh quan sat duoc tai t-1, lay top-K.
pool = load_df(CACHE / "pool_universe.parquet")
if pool is None:
    print("Chua co pool_universe.parquet (meme + neutral gop lai).")
else:
    pool = pool.sort_values(["ticker", "date"])
    memeset = set(bt["ticker"].unique())
    cl = pool.pivot(index="date", columns="ticker", values="close")
    vo = pool.pivot(index="date", columns="ticker", values="vol")
    hi = pool.pivot(index="date", columns="ticker", values="high")
    lo = pool.pivot(index="date", columns="ticker", values="low")
    op = pool.pivot(index="date", columns="ticker", values="open")
    dv = cl * vo

    # ---- MOI feature deu .shift(1): chi dung thong tin da BIET tai t-1 ----
    dv_s, cl_s, hi_s, lo_s = dv.shift(1), cl.shift(1), hi.shift(1), lo.shift(1)
    tr_ = pd.concat([hi_s-lo_s, (hi_s-cl_s.shift(1)).abs(), (lo_s-cl_s.shift(1)).abs()]).groupby(level=0).max()
    atr5, atr60 = tr_.rolling(5).mean(), tr_.rolling(60).mean()
    f_vol   = dv_s.rolling(5).mean() / dv_s.rolling(60).mean()   # dong tien tang toc
    f_volx  = (atr5/cl_s) / (atr60/cl_s)                         # vola no ra
    f_ret20 = cl_s/cl_s.shift(20) - 1                            # suc manh gan day
    f_cheap = -np.log(cl_s.clip(lower=.01))                      # gia thap = de thanh meme
    zs = lambda d: d.sub(d.mean(axis=1), axis=0).div(d.std(axis=1).replace(0, np.nan), axis=0)
    SCORE = (zs(f_vol).fillna(0) + zs(f_volx).fillna(0)
             + .5*zs(f_ret20).fillna(0) + .5*zs(f_cheap).fillna(0))
    SCORE = SCORE.where(dv_s.rolling(20).mean() >= 1e6)          # thanh khoan toi thieu, cung t-1
    RANK  = SCORE.rank(axis=1, ascending=False)
    print(f"Pool {cl.shape[1]} ma | {cl.shape[0]} ngay | score point-in-time da san sang")
    print("Watchlist hom nay (top 15 theo diem t-1):")
    last = RANK.iloc[-1].dropna().sort_values().head(15)
    print("  " + ", ".join(f"{t}" for t in last.index))
    print("""
KET QUA DA CHAY (xem bao cao):
  All 195 ma (khong chon)     exp +0.70%  Sharpe 0.55  maxDD -75.4%
  HINDSIGHT seed 82 ma        exp +2.92%  Sharpe 0.81  maxDD -48.6%   <- BIAS
  PIT selector TOP-30         exp +1.00%  Sharpe 0.59  maxDD -26.1%   <- hop le
  Ngau nhien 30 ma/ngay       exp -0.55%  Sharpe -0.16                <- control

PHA LOANG POOL (mo phong base-rate that):
  42% meme -> exp +1.00%   26% -> +0.33%   12% -> -0.44%   4% -> -0.49%
  => Selector chi 'gioi' khi pool da giau san ten meme. O ty le thuc te (~1-4%)
     feature GIA/VOLUME thuan tuy KHONG tim ra duoc meme. Thu con thieu chinh la
     SOCIAL/BORROW real-time - thu khong co lich su mien phi de backtest.""")
